In [0]:
pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wyattowalsh/basketball")

print("Path to dataset files:", path)

In [0]:
#Check available files: 
import os

print("Files downloaded:")
for f in os.listdir(path):
    file_path = os.path.join(path, f)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"{f} - {size_mb:.2f} MB")

In [0]:
# Download the SQLite file to check schema 

# Imports
import kagglehub
import shutil
import os
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit, sha2, concat_ws, coalesce, col
from pyspark.sql.types import NumericType
import re
from pyspark.sql.functions import regexp_extract, col

#Configuration
KAGGLE_DATASET = "wyattowalsh/basketball"
CATALOG = "nba_data_kaggle"
SCHEMA = "bronze"
VOLUME = "bronze_files"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"


# Download the latest version of the dataset from Kaggle
kaggle_path = kagglehub.dataset_download(KAGGLE_DATASET, path="nba.sqlite")
print("Downloaded to temporary cluster path:", kaggle_path)

# Generate a timestamp to append to the filename, so each day's
# download lands as a distinct file in the volume
download_date = datetime.now().strftime("%Y%m%d")

source_file = kaggle_path
destination_file = os.path.join(VOLUME_PATH, f"nba_data_kaggle_{download_date}.sqlite")

# Copy from the temporary cluster cache into the Unity Catalog managed volume,
# so the file persists beyond the current cluster session
shutil.copy(source_file, destination_file)

print("File copied to:", destination_file)

In [0]:
# =============================================================================
# CELL: Verify volume copy integrity
# =============================================================================
import os

volume_file = "/Volumes/nba_data_kaggle/bronze/bronze_files/nba_data_kaggle_20260809.sqlite"

size_mb = os.path.getsize(volume_file) / (1024 * 1024)
print(f"File size in volume: {size_mb:.2f} MB")

In [0]:
# =============================================================================
# CELL: Explore NBA SQLite source schema
# Purpose: Inventory tables, row counts, and columns BEFORE deciding grain
#          for Bronze ingestion. Read-only — no writes.
# =============================================================================

import sqlite3

sqlite_path = "/Volumes/nba_data_kaggle/bronze/bronze_files/nba_data_kaggle_20260809.sqlite"

conn = sqlite3.connect(sqlite_path)
cursor = conn.cursor()

# Get all table names (exclude SQLite internal tables)
cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""")
tables = [row[0] for row in cursor.fetchall()]

print(f"Found {len(tables)} tables:\n")

schema_summary = []

for table in tables:
    # Row count
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    row_count = cursor.fetchone()[0]

    # Column info: cid, name, type, notnull, default_value, pk
    cursor.execute(f"PRAGMA table_info({table});")
    columns = cursor.fetchall()
    col_names = [col[1] for col in columns]
    pk_cols = [col[1] for col in columns if col[5] > 0]  # pk flag

    schema_summary.append({
        "table": table,
        "row_count": row_count,
        "num_columns": len(columns),
        "pk_columns": pk_cols,
        "columns": col_names
    })

    print(f"{table:<25} rows={row_count:>10,}  cols={len(columns):>3}  pk={pk_cols}")

conn.close()

In [0]:
# =============================================================================
# CELL: Diagnose grain candidates for key tables
# Purpose: Print full column lists for tables we'll need in Silver, and test
#          candidate keys for uniqueness (COUNT(*) vs COUNT(DISTINCT key)).
#          Still read-only — confirming grain before any Bronze code.
# =============================================================================

import sqlite3

sqlite_path = "/Volumes/nba_data_kaggle/bronze/bronze_files/nba_data_kaggle_20260809.sqlite"

conn = sqlite3.connect(sqlite_path)
cursor = conn.cursor()

tables_to_inspect = ["game", "game_info", "game_summary", "line_score", "team", "team_history"]

for table in tables_to_inspect:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table}")
    print('=' * 60)

    cursor.execute(f"PRAGMA table_info({table});")
    columns = cursor.fetchall()
    for col in columns:
        # col = (cid, name, type, notnull, default_value, pk)
        print(f"  {col[1]:<25} {col[2]}")

# -----------------------------------------------------------------------
# Candidate key uniqueness checks
# Adjust candidate_keys dict below if column names differ from what we expect
# -----------------------------------------------------------------------
print(f"\n{'=' * 60}")
print("CANDIDATE KEY UNIQUENESS CHECKS")
print('=' * 60)

candidate_keys = {
    "game": ["game_id"],
    "game_info": ["game_id"],
    "game_summary": ["game_id"],
    "line_score": ["game_id"],
    "team": ["id"],
    "team_history": ["team_id"],  # likely needs a second column (year) to be unique
}

for table, keys in candidate_keys.items():
    key_expr = ", ".join(keys)
    cursor.execute(f"SELECT COUNT(*), COUNT(DISTINCT {key_expr}) FROM {table};")
    total, distinct = cursor.fetchone()
    status = "UNIQUE" if total == distinct else "NOT UNIQUE (duplicates exist)"
    print(f"  {table:<15} key=({key_expr}):  total={total:>10,}  distinct={distinct:>10,}  → {status}")

conn.close()